# ICD Navigator · 医码当先系统说明书

一个针对不规范诊断文本的ICD-10编码映射系统

## 一、项目背景与问题定义

### 1. 应用场景

本系统面向医院病案室工作人员、医保审核人员、临床数据管理人员等医疗相关从业者，以及需要将口语化诊断描述批量转换为标准 ICD-10 编码的科研人员和数据分析师。

### 2. 当前痛点

- **人工编码效率低**：医生书写的诊断描述多为口语化表达（如"乳房恶性肿瘤""肺癌术后""高血压3级""2型糖尿病"），病案室工作人员需要逐条翻阅 ICD-10 编码手册进行手工匹配，一条诊断的编码耗时约 30 秒至 1 分钟，大批量处理时效率极低。
- **编码准确率依赖个人经验**：ICD-10 编码体系复杂（全疾病编码包含 22 个章节、数千个三位码），编码人员水平参差不齐，容易出现编码错误或遗漏。
- **现有工具智能化不足**：传统 ICD 编码查询工具多基于关键词精确匹配，无法处理口语化表达、同义词、术后/转移/分期等复杂描述，缺乏语义理解能力。
- **企业级 API 调用不稳定**：单一 API 模型存在限流、超时、返回格式不稳定等问题，缺乏容错机制。

### 3. 项目目标

- 利用大语言模型（LLM）的语义理解能力，实现口语化诊断描述到标准 ICD-10 编码的自动转换
- 构建多模型轮转策略，保障 API 调用的稳定性和可用性
- 采用向量检索 + 精排作为兜底方案，提升编码覆盖率
- 提供友好的 Web 界面，降低使用门槛，实现"上传即用、一键下载"

### 4. 边界说明

- **支持 ICD-10 全疾病编码**（A00-Z99），覆盖全部 22 个章节，包括传染病、肿瘤、循环系统、呼吸系统、消化系统、内分泌、神经系统等所有疾病类别
- **不支持 ICD-10 四位码细分**，仅输出三位码（如 C50 而非 C50.9，I10 而非 I10.x）
- **不支持多诊断联合编码**，每条诊断仅输出一个最佳匹配的 ICD 编码
- **不支持诊断文本的质量校验**，如输入包含明显错误或歧义的描述，系统可能给出不准确的编码
- **暂不支持历史记录查询和用户管理**等高级功能
- **准确率受知识库覆盖范围影响**：罕见病、非常规表述的诊断可能编码准确率较低

## 二、Demo 功能范围

### 1. 输入形式

| 输入类型 | 格式要求 | 示例 |
|----------|----------|------|
| CSV 文件上传 | UTF-8 编码，包含至少一列诊断描述 | 单列或多列 CSV，如"诊断描述""患者ID"等 |
| API 密钥 | 文本输入框，支持密码遮蔽 | 用户自有的 LLM API 密钥 |
| 参数配置 | 界面滑块和数字输入 | 置信度阈值（0~1）、检索候选数（5~50） |
| 示例数据加载 | 一键按钮 | 内置 30 条覆盖多章节的诊断示例 |

### 2. 输出形式

| 输出类型 | 内容 | 格式 |
|----------|------|------|
| 主结果文件 | 原始数据 + ICD-10 编码 + 置信度 + 匹配方式 + 处理状态 | CSV 文件 |
| 处理日志 | 每条诊断的 Step 1~3 详细结果 | JSON 文件 |
| 统计摘要 | 序号、诊断、编码、置信度、处理时间等汇总 | CSV 文件 |
| 前端展示 | 实时进度条、处理统计、结果预览表格 | Web 界面 |

### 3. 核心功能

| 功能模块 | 说明 |
|----------|------|
| API 密钥验证 | 用户输入密钥后一键验证可用性，验证结果显示在下方 |
| 文件上传与预览 | 支持 CSV 上传，自动读取表头，可预览前 10 条数据 |
| 示例数据加载 | 内置覆盖多章节的示例数据集，无需准备文件即可快速体验 |
| 诊断列选择 | 下拉框自动列出所有列名，用户选择诊断描述所在列 |
| 参数可配置 | 置信度阈值和检索候选数可通过界面调整 |
| 知识库下载 | 一键下载 ICD-10 全疾病编码知识库文件（CSV 格式） |
| 三步编码处理 | Step 1 LLM 提取 → Step 2 向量检索 → Step 3 精排确认 |
| 多模型轮转 | Primary 模型失败自动切换，Secondary 兜底，成功重置 |
| 进度可视化 | 进度条 + 实时状态文字 + 当前模型显示 + 处理统计 |
| 结果下载 | 一键打包 ZIP 下载，含主结果、日志、摘要 |

### 4. 功能边界

- 不支持自定义知识库上传
- 不支持批量文件处理（一次只能处理一个 CSV）
- 不支持结果的历史记录和对比
- 不支持断点续传（刷新页面后需重新处理）
- 不支持离线运行（依赖云端 API）
- 准确率受疾病类别影响：常见疾病（高血压、糖尿病等）准确率高，罕见病和复杂诊断准确率相对较低

## 三、系统设计

### 1. 系统架构

![系统架构示意图](img/Ai-agent%20系统架构.png)

### 2. 模块说明

| 模块 | 输入 | 处理逻辑 | 输出 |
|------|------|----------|------|
| **app.py** | 用户操作（密钥、文件、参数） | 管理 Session State，协调各模块调用，渲染 UI | 界面展示、结果下载 |
| **config.py** | — | 存储全局配置（API 地址、模型列表、检索参数） | 配置常量 |
| **icd_processor.py** | CSV 诊断数据 | 加载全疾病知识库 → 构建向量索引 → 逐条三步编码处理 | 编码结果字典 |
| **utils/api_client.py** | API 请求参数 | 封装 HTTP 请求，实现多模型轮转和错误处理 | API 响应结果 |
| **utils/logger.py** | 日志消息 | 配置日志文件和控制台输出 | 日志文件 |
| **utils/helpers.py** | 文件路径、列名等 | 列名生成、ZIP 打包、文件验证等辅助功能 | 处理后的路径/文件 |

### 3. 执行流程

![执行流程示意图](img/Ai-agent%20执行流程.png)

# 模型调用指南

截止2026年6月，文本生成模型的华为官方报价如下（[MaaS模型服务计费项](https://support.huaweicloud.com/price-maas/price-maas-0001.html)）：
| 模型名称 | 模型优缺点 | 计费子项 | 单价（元/千Tokens） |
| :--- | :--- | :--- | :--- |
| **DeepSeek-V4-Pro** | **优点：** 旗舰级综合能力，推理、代码、多语言表现顶尖，上下文处理出色。<br>**缺点：** 定价较高，复杂推理时输出成本上升明显。 | 输入 | 0.012 |
| | | 输出 | 0.024 |
| **Kimi-K2.6** | **优点：** 输入极便宜，适合长文档分析、知识库问答等输入密集型任务。<br>**缺点：** 输出价格偏高，长生成场景成本不占优。 | 输入 | 0.0065 |
| | | 输出 | 0.027 |
| **DeepSeek-V4-Flash** | **优点：** 价格极具竞争力，适合高并发、简单任务批量处理。<br>**缺点：** 复杂推理和生成质量低于Pro版本。 | 输入 | 0.001 |
| | | 输出 | 0.002 |
| **DeepSeek-R1-0528** | **优点：** 强推理模型，数学、逻辑等复杂推理任务表现优秀。<br>**缺点：** 输出价格较高，推理链较长时成本增加明显。 | 输入 | 0.004 |
| | | 输出 | 0.016 |
| **DeepSeek-V3** | **优点：** 性价比高，通用能力强，适合日常对话和内容生成。<br>**缺点：** 在顶级复杂推理上弱于R1/Pro系列。 | 输入 | 0.002 |
| | | 输出 | 0.008 |
| **DeepSeek-V3.1** | **优点：** V3升级版，综合能力提升，保持良好性价比。<br>**缺点：** 相比V3价格有所上升，输出成本翻倍。 | 输入 | 0.004 |
| | | 输出 | 0.012 |
| **DeepSeek-V3.2** | **优点：** 输出价格大幅优化，适合输出密集型应用如文章生成。<br>**缺点：** 极致的输出成本可能在某些复杂任务上能力有所取舍。 | 输入 | 0.002 |
| | | 输出 | 0.003 |
| **Qwen3-235B-A22B** | **优点：** 超大参数MoE，知识广博，思考模式推理能力强。<br>**缺点：** 思考模式输出成本高，模型体量大可能影响响应速度。 | 输入 | 0.002 |
| | | 输出 - 思考模式 | 0.02 |
| | | 输出 - 非思考模式 | 0.008 |
| **Qwen3-30B-A3B** | **优点：** 输入极便宜，轻量高效，适合高并发简单任务。<br>**缺点：** 模型规模较小，复杂任务能力有限。 | 输入 | 0.00075 |
| | | 输出 - 思考模式 | 0.0075 |
| | | 输出 - 非思考模式 | 0.003 |
| **Qwen3-32B** | **优点：** 密集模型，32B参数下性能出色，推理部署相对友好。<br>**缺点：** 思考模式输出价差大，大规模使用需控制思考模式调用。 | 输入 | 0.002 |
| | | 输出 - 思考模式 | 0.02 |
| | | 输出 - 非思考模式 | 0.008 |
| **LongCat-Flash-Chat** | **优点：** 价格均衡，适合日常对话和通用文本处理。<br>**缺点：** 品牌知名度较低，生态和社区支持可能有限。 | 输入 | 0.002 |
| | | 输出 | 0.008 |
| **GLM-5**<br>（单次请求输入Token<32K） | **优点：** 国产开源旗舰，中文理解与生成出色，短文本性价比高。<br>**缺点：** 长文本场景价格上浮明显，输出成本整体偏高。 | 输入 | 0.004 |
| | | 输出 | 0.018 |
| **GLM-5**<br>（单次请求输入Token≥32K） | **优点：** 原生支持超长上下文，长文档处理能力强。<br>**缺点：** 长文本使用时输入输出成本均显著增加。 | 输入 | 0.006 |
| | | 输出 | 0.022 |
| **GLM-5.1**<br>（单次请求输入Token<32K） | **优点：** GLM-5升级版，综合性能提升。<br>**缺点：** 价格较GLM-5全面上涨，性价比需根据效果评估。 | 输入 | 0.006 |
| | | 输出 | 0.024 |
| **GLM-5.1**<br>（单次请求输入Token≥32K） | **优点：** 长文本场景下性能增强。<br>**缺点：** 定价在同类中最高，适合对质量要求极高且预算充足的场景。 | 输入 | 0.008 |
| | | 输出 | 0.028 |

## glm-5

In [ ]:
import requests
import json

if __name__ == '__main__':
    url = "https://api.modelarts-maas.com/v2/chat/completions"  # API地址
    api_key = "MAAS_API_KEY"  # 把MAAS_API_KEY替换成已获取的API Key

    # Send request.
    headers = {
        'Content-Type': 'application/json',
        'Authorization': f'Bearer {api_key}'
    }
    data = {
        "model": "glm-5",  # model参数
        "messages": [
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": "你好"}
        ]
    }
    response = requests.post(url, headers=headers, data=json.dumps(data), verify=False)

    # Print result.
    print(response.status_code)
    print(response.text)

## glm-5.1

In [ ]:
import requests
import json

if __name__ == '__main__':
    url = "https://api.modelarts-maas.com/v2/chat/completions"  # API地址
    api_key = "MAAS_API_KEY"  # 把MAAS_API_KEY替换成已获取的API Key

    # Send request.
    headers = {
        'Content-Type': 'application/json',
        'Authorization': f'Bearer {api_key}'
    }
    data = {
        "model": "glm-5.1",  # model参数
        "messages": [
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": "你好"}
        ]
    }
    response = requests.post(url, headers=headers, data=json.dumps(data), verify=False)

    # Print result.
    print(response.status_code)
    print(response.text)

## kimi-k2.6

In [ ]:
import requests
import json

if __name__ == '__main__':
    url = "https://api.modelarts-maas.com/v2/chat/completions"  # API地址
    api_key = "MAAS_API_KEY"  # 把MAAS_API_KEY替换成已获取的API Key

    # Send request.
    headers = {
        'Content-Type': 'application/json',
        'Authorization': f'Bearer {api_key}'
    }
    data = {
        "model": "kimi-k2.6",  # model参数
        "messages": [
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": "你好"}
        ]
    }
    response = requests.post(url, headers=headers, data=json.dumps(data), verify=False)

    # Print result.
    print(response.status_code)
    print(response.text)

## qwen3-235b-a22b

In [ ]:
import requests
import json

if __name__ == '__main__':
    url = "https://api.modelarts-maas.com/v2/chat/completions"  # API地址
    api_key = "MAAS_API_KEY"  # 把MAAS_API_KEY替换成已获取的API Key

    # Send request.
    headers = {
        'Content-Type': 'application/json',
        'Authorization': f'Bearer {api_key}'
    }
    data = {
        "model": "qwen3-235b-a22b",  # model参数
        "messages": [
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": "你好"}
        ]
    }
    response = requests.post(url, headers=headers, data=json.dumps(data), verify=False)

    # Print result.
    print(response.status_code)
    print(response.text)

## deepseek-v3.2

In [ ]:
import requests
import json

if __name__ == '__main__':
    url = "https://api.modelarts-maas.com/v2/chat/completions"  # API地址
    api_key = "MAAS_API_KEY"  # 把MAAS_API_KEY替换成已获取的API Key

    # Send request.
    headers = {
        'Content-Type': 'application/json',
        'Authorization': f'Bearer {api_key}'
    }
    data = {
        "model": "deepseek-v3.2",  # model参数
        "messages": [
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": "你好"}
        ],
       "thinking": {
            "type": "enabled"  # 是否开启深度思考模式，默认关闭
        }
    }
    response = requests.post(url, headers=headers, data=json.dumps(data), verify=False)

    # Print result.
    print(response.status_code)
    print(response.text)

## deepseek-v4-pro

In [ ]:
import requests
import json

if __name__ == '__main__':
    url = "https://api.modelarts-maas.com/v2/chat/completions"  # API地址
    api_key = "MAAS_API_KEY"  # 把MAAS_API_KEY替换成已获取的API Key

    # Send request.
    headers = {
        'Content-Type': 'application/json',
        'Authorization': f'Bearer {api_key}'
    }
    data = {
        "model": "deepseek-v4-pro",  # model参数
        "messages": [
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": "你好"}
        ]
    }
    response = requests.post(url, headers=headers, data=json.dumps(data), verify=False)

    # Print result.
    print(response.status_code)
    print(response.text)

## deepseek-v4-flash

In [ ]:
import requests
import json

if __name__ == '__main__':
    url = "https://api.modelarts-maas.com/v2/chat/completions"  # API地址
    api_key = "MAAS_API_KEY"  # 把MAAS_API_KEY替换成已获取的API Key

    # Send request.
    headers = {
        'Content-Type': 'application/json',
        'Authorization': f'Bearer {api_key}'
    }
    data = {
        "model": "deepseek-v4-flash",  # model参数
        "messages": [
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": "你好"}
        ]
    }
    response = requests.post(url, headers=headers, data=json.dumps(data), verify=False)

    # Print result.
    print(response.status_code)
    print(response.text)

## bge-m3

In [ ]:
import requests
import json

if __name__ == '__main__':
    url = "https://api.modelarts-maas.com/v1/embeddings"  # API地址
    api_key = "MAAS_API_KEY"  # 把MAAS_API_KEY替换成已获取的API Key

    # Send request.
    headers = {
        'Content-Type': 'application/json',
        'Authorization': f'Bearer {api_key}'
    }

    texts = ["这是一只小猫", "这是一只小狗"]
    data = {
        "model": "bge-m3",  # model参数
        "input": texts,  # input类型可为string or string[]
        "encoding_format": "float"  # 取值范围："float","base64"
    }
    response = requests.post(url, headers=headers, data=json.dumps(data), verify=False)

    # Print result.
    print(response.status_code)
    print(response.text)

## bge-reranker-v2-m3

In [ ]:
import requests
import json

if __name__ == '__main__':
    url = "https://api.modelarts-maas.com/v1/rerank"  # API地址
    api_key = "MAAS_API_KEY"  # 把MAAS_API_KEY替换成已获取的API Key

    # Send request.
    headers = {
        'Content-Type': 'application/json',
        'Authorization': f'Bearer {api_key}'
    }
    data = {
        "model": "bge-reranker-v2-m3",
        "query": "牛是一种动物如何冲泡一杯好喝的咖啡？",  # input类型可为string或string[]。
        "documents": [
            "咖啡豆的产地主要分布在赤道附近，被称为‘咖啡带’。",
            "法压壶的步骤：1. 研磨咖啡豆。2. 加入热水。3. 压下压杆。4. 倒入杯中。",
            "意式浓缩咖啡需要一台高压机器，在9个大气压下快速萃取。",
            "挑选咖啡豆时，要注意其烘焙日期，新鲜的豆子风味更佳。",
            "手冲咖啡的技巧：控制水流速度、均匀注水和合适的水温（90-96°C）是关键。"
        ]
    }

    response = requests.post(url, headers=headers, data=json.dumps(data), verify=False)

    # Print result.
    print(response.status_code)
    print(response.text)

# ICD10文库构建

## 剪裁文件

In [13]:
import pandas as pd
import os

# 确保目标目录存在
output_dir = r'data\icd10_data\test'
os.makedirs(output_dir, exist_ok=True)

# 读取Excel文件的"总表"sheet
df = pd.read_excel(r'data\icd10_data\raw\ICD-10医保1.0版.xlsx', sheet_name='总表', dtype={'疾病编码': str})

# 删除“章编码”是“附录”的行
df = df[df['章编号'] != '附录']

# 保存为CSV文件
df.to_csv(os.path.join(output_dir, 'ICD-10医保1.0版.总表.csv'), index=False)

## 保存三位码

In [14]:
import pandas as pd
import os

# 读取之前保存的CSV文件
input_file = r'data\icd10_data\test\ICD-10医保1.0版.总表.csv'
df = pd.read_csv(input_file)

# 只保留指定的列
columns_to_keep = ['章编号', '章名称', '节编码', '节名称', '三位码', '三位名称']
df_filtered = df[columns_to_keep]

# 去重
df_unique = df_filtered.drop_duplicates()

# 保存到同目录
output_file = os.path.join(os.path.dirname(input_file), 'ICD-10医保1.0版.总表.三位码.csv')
df_unique.to_csv(output_file, index=False)

print(f"处理完成！去重后共 {len(df_unique)} 行数据")
print(f"文件已保存至: {output_file}")

处理完成！去重后共 2048 行数据
文件已保存至: data\icd10_data\test\ICD-10医保1.0版.总表.三位码.csv


## 规整三位码

In [15]:
import pandas as pd

file_path = r"data\icd10_data\test\ICD-10医保1.0版.总表.三位码.csv"

df = pd.read_csv(file_path, dtype={'三位码': str})
df['三位码'] = df['三位码'].str.strip()
# mask = df['三位码'].str.match(r'^[A-Z]\d{2}[^A-Za-z0-9]$', na=False)
mask = df['三位码'].str.match(r'^[A-Z]\d{2}[^A-Za-z0-9]$', na=False)
df.loc[mask, '三位码'] = df.loc[mask, '三位码'].str[:3]
df.to_csv(file_path, index=False, encoding='utf-8-sig')

df = pd.read_csv(file_path, dtype={'三位码': str})
codes = df['三位码'].dropna().str.strip()
invalid = codes[~codes.str.match(r'^[A-Z]\d{2}$')]

if len(invalid) == 0:
    print("所有“三位码”都符合格式要求")
else:
    for code in invalid:
        print(code)

所有“三位码”都符合格式要求


## 补充别名

In [ ]:
import pandas as pd
import requests
import json
import time
import os
from tqdm.notebook import tqdm
import urllib3
urllib3.disable_warnings()

# ============ 配置 ============
API_URL = "https://api.modelarts-maas.com/v2/chat/completions"
API_KEY = "MY_KEY"
INPUT_PATH = r"data\icd10_data\test\ICD-10医保1.0版.总表.三位码.csv"
OUTPUT_PATH = r"data\icd10_data\test\ICD-10医保1.0版.总表.三位码.别名.csv"
BATCH_SIZE = 50

# ============ 读取数据 ============
df = pd.read_csv(INPUT_PATH)

# 如果"别名"列不存在则新建，空值填充为空字符串
if "别名" not in df.columns:
    df["别名"] = ""
df["别名"] = df["别名"].fillna("")

# 只获取别名为空的疾病名称
empty_alias_mask = df["别名"] == ""
disease_names = df.loc[empty_alias_mask, "三位名称"].dropna().unique().tolist()
print(f"共 {len(disease_names)} 个需要填充的疾病名称")

# ============ 调用大模型批量生成别名 ============
aliases_dict = {}

batches = range(0, len(disease_names), BATCH_SIZE)
for i in tqdm(batches, desc="处理批次"):
    batch = disease_names[i:i+BATCH_SIZE]
    
    disease_list = "\n".join(batch)
    prompt = f"""你是一个医学文本处理助手。请为以下ICD-10疾病标准名称生成常见别名，包括中文口语化表达和正式/非正式英文缩写。

要求：
1. 每行输出一个疾病的标准名称和所有别名，用竖线"|"分隔
2. 别名包括：中文口语简称、俗称、患者常用说法、以及正式和非正式的英文缩写（如LC、GC等）
3. 多个别名用中文逗号分隔
4. 如果确实没有常见别名或缩写，别名部分留空
5. 不要输出任何解释，只输出结果行

示例输入：
乳房恶性肿瘤
支气管或肺恶性肿瘤
胃恶性肿瘤

示例输出：
乳房恶性肿瘤|乳腺癌,乳癌,乳房癌,BC
支气管或肺恶性肿瘤|肺癌,支气管癌,LC
胃恶性肿瘤|胃癌,GC

现在请处理以下疾病名称：
{disease_list}"""
    
    headers = {
        'Content-Type': 'application/json',
        'Authorization': f'Bearer {API_KEY}'
    }
    data = {
        "model": "deepseek-v3.2",
        "messages": [
            {"role": "system", "content": "你是一个专业的医学文本处理助手，严格按格式输出。熟悉中英文医学常用缩写。"},
            {"role": "user", "content": prompt}
        ],
        "temperature": 0
    }
    
    response = requests.post(API_URL, headers=headers, json=data, verify=False)
    
    if response.status_code == 200:
        result = response.json()
        content = result["choices"][0]["message"]["content"]
        
        for line in content.strip().split("\n"):
            line = line.strip()
            if "|" in line:
                parts = line.split("|", 1)
                if len(parts) == 2:
                    name, alias = parts
                    aliases_dict[name.strip()] = alias.strip()
    
    time.sleep(1)

# ============ 合并别名到原数据（仅更新空别名） ============
for name, alias in aliases_dict.items():
    mask = (df["三位名称"] == name) & (df["别名"] == "")
    df.loc[mask, "别名"] = alias

os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")

# ============ 输出统计信息 ============
total = len(df)
has_alias = (df["别名"] != "").sum()
print(f"\n处理完成！")
print(f"总记录数: {total}")
print(f"有别名记录: {has_alias}")
print(f"别名覆盖率: {has_alias/total*100:.1f}%")
print(f"结果已保存至 {OUTPUT_PATH}")

共 87 个需要填充的疾病名称


处理批次:   0%|          | 0/2 [00:00<?, ?it/s]


处理完成！
总记录数: 2048
有别名记录: 1999
别名覆盖率: 97.6%
结果已保存至 data\icd10_data\test\ICD-10医保1.0版.总表.三位码.别名3.csv


## 检查结果

In [9]:
import pandas as pd

df = pd.read_csv("data/icd10_data/test/ICD-10医保1.0版.总表.三位码.别名.csv")
empty_alias = df[df['别名'].isna()]
empty_alias.to_csv("data/icd10_data/test/ICD-10医保1.0版.总表.三位码.无别名.csv", index=True)